# Create database and JSON ssot from ODM

```python3 webfillallin1.py [path to your]/IM de FORCE```

In [ ]:
#odm_source_folder = '/Users/bue/dev/fyyccim-refmodels/riddle/IM'
#odm_source_folder = '/Users/bue/dev/fyyccim-refmodels/PIM/IM'
#odm_source_folder = '/Users/bue/dev/fyyccim-refmodels/CRM/IM'
#odm_source_folder = '/Users/bue/dev/fyyccim-health/Health/IM'
odm_source_folder = '/Users/bue/BOPT_IM_Stand stb 20201021/'
odm_source_folder = '/Users/bue/dev/borr/2021-06-25 BORR IM/IM'
odm_source_folder = '/Users/bue/dev/geberit/DEAP/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-refmodels-master/CRM/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-refmodels-master/CRM/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-refmodels/raetsel-3lang/IM'
odm_source_folder = '/Users/bue/dev/borr/2021-07-14 BORR IM/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-im/ModellModell/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-projects/IM'
odm_source_folder = '/Users/bue/dev/borr/2021-08-18 BORR IM'
odm_source_folder = '//Users/bue/projects/agravis/IM'

In [ ]:
import sys
import os
from pathlib import Path
import glob

In [ ]:
base_path = Path(odm_source_folder)
assert os.path.isdir(odm_source_folder), "Cannot find source folder {}".format(odm_source_folder)
project_files = list(base_path.glob('*.dmd'))
assert len(project_files) == 1, "Cannot find exactly 1 ODM .dmd file in source folder {}: {}".format(odm_source_folder, project_files)
from IPython.core.display import HTML
HTML('<span style="font-family: Impact; font-size:48px">Processing the information model in <span style="color: darkorange">{0}</span></span>'.format(odm_source_folder))

In [ ]:
import logging

os.makedirs('log', exist_ok=True)
logfile = 'log/odm2ssot.log'

handler = logging.handlers.RotatingFileHandler(logfile, maxBytes=(1024*1024*10), backupCount=10)
formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
handler.setLevel(logging.DEBUG)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)

In [ ]:
# Add toolbox to python library path
sys.path.insert(0, os.path.abspath('../../pythonWork/pythonSource'))
sys.path.insert(1, os.path.abspath('../../pythonWork/pythonSource/IM_db'))
from IM_db.IM_JSON import JSModel
from IM_WEB.IM_HTML import entityenviron
from IM_DB import parameters

In [ ]:
from IM_DB import logmessages

def tap_logmessages(message: str):
    logger.warning(message)

logmessages.logtap = tap_logmessages

## Initialize parameters

In [ ]:
parameters.initparam(odm_source_folder)

## Create database + JSON

In [ ]:
import contextlib
os.makedirs(parameters.dbDirect(), exist_ok=True)

with contextlib.suppress(FileNotFoundError):
    os.remove(parameters.dbFilePath())
    print(f"Deleted existing DB {parameters.dbFilePath()}")

In [ ]:
from IM_ODM import fillDB
print('Creating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))
try:
    fillDB.filldbmain(odm_source_folder, createnewdb=True)
except:
    print('Consult logfile {}'.format(parameters.logfilepath()))
    raise

In [ ]:
def fillDB_profile():
    fillDB.filldbmain(odm_source_folder, createnewdb=True)

In [ ]:
#import cProfile
#cProfile.run('fillDB_profile()')

## Find and validate the database

In [ ]:
database_file = os.path.abspath(parameters.dbFilePath())
assert os.path.isfile(database_file)
'Working with database {}'.format(database_file)

In [ ]:
from IM_DB import dbConnect
from IM_JSON import sql2json,JSModel
    
dbConnect.openDB(parameters.dbFilePath(), pfks='ON')
jsmodel = JSModel(pmodel=sql2json(pdbname=parameters.dbFilePath()))
# printHTML.setWebDirec(p_webdirec=None)
# listWebdoku.listwebmain(plang=Languagetext.reportLang(),pmodel=jsmodel)
json_file = jsmodel.printmodel(pfilepath=parameters.dbDirect(), pfilename=parameters.odmModelName())
dbConnect.closeDB()
'Generated {}'.format(json_file)

# Verification and validation

In [ ]:
import json

with open(json_file, 'r') as source:
    reload = json.load(source)

In [ ]:
for root in reload:
    print('Root {} contains {}'.format(root, len(reload[root])))

In [ ]:
reload['languages']

## Add git revison to JSON file if available

This requires [git](https://git-scm.com/downloads)

Used command:
`git --git-dir=repo/.git describe --always`

Persist with git label added to _imprint_

In [ ]:
from pathlib import Path

start = Path(odm_source_folder).resolve()
print('Scanning {}'.format(start))
def get_git_dir(start: Path, limit: int = 3) -> Path:
    path = start.absolute()
    while path and limit > 0:
        candidates = path.glob('.git')
        for folder in candidates:
            if folder.is_dir():
                return folder
        path = path.parent
        limit -= 1
    return None

gitfolder = get_git_dir(start)
if gitfolder:
    print("git repo located in {}".format(str(gitfolder)))

In [ ]:
import subprocess
#label = '27c6d8b91c0'
label = 'e478547f8af'

if gitfolder:
    label = subprocess.check_output(['git', '--git-dir=' + str(gitfolder), 'describe', '--always']).decode().strip()

if label:  
    print("Found git revision {0} checked out in {1}".format(label, gitfolder))
    reload['_imprint_']['model-git-revision'] = label
    with open(json_file, 'w') as output:
        json.dump(reload, output, indent=2, sort_keys=False)
    print("Wrote extended _imprint_ to {}".format(json_file))

In [ ]:
HTML('<span style="font-family: Impact; font-size:48px">SSOT ready <span style="color: darkgreen">{0}</span></span>'.format(json_file))

In [ ]:
reload['model']

In [ ]:
reload['_imprint_']

In [ ]:
with open(json_file, 'r') as final_json:
    verification = json.load(final_json)
    
print(f"Summary for {json_file}")

for base_key in verification:
    print("Topic {0} contains {1} entries: ".format(base_key, len(reload[base_key])))

In [ ]:
verification['model']

In [ ]:
verification['_imprint_']